In [ ]:
library(tidyverse)
library(scales)
library(fs)
library(phylotools)

###
Mode <- function(x) {
  ux <- unique(x)
  ux[which.max(tabulate(match(x, ux)))]
}

library(metacoder)
options(ENTREZ_KEY='94ef22f4482d7bc124b52e1b9036688eb608')



In [ ]:



## set paths first here!
workingpath <- getwd()
dataobjpath <- str_c("/hpc/projects/balla_group/sra_experiments/versioned_zf_output/75k_unstable/adata_obj")
salmonpath <- str_c("/hpc/projects/balla_group/sra_experiments/versioned_zf_output/75k_unstable/salmon_steps")

outpath0 <- str_c("/hpc/projects/balla_group/sra_experiments/versioned_zf_output/75k_unstable/metatranscriptome_all")


outpath <- str_c(outpath0, "/RNAquarium_outputs")
outpathvirus <- str_c(outpath, "/virus_outputs")


In [ ]:
## NOTES
#we want to use taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_2025-10-09.tsv
## but restrict to fish-associated using same code as in 
## this is most recent october version (version from september is used in Salmon counts)


In [ ]:
setwd(outpathvirus)

allchunks_diamondnr_andblastntclustered_viruses <- read_tsv("taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_2025-10-09.tsv")             

In [ ]:
## change wd back
setwd(workingpath)

In [ ]:
thresholded_hit_virusesabundantfewer <- allchunks_diamondnr_andblastntclustered_viruses %>% dplyr::filter(viruscategorysimple_NTorNR != "Adapter")


thresholded_hit_virusesabundantfewer$prjtest <- NULL
n_distinct(thresholded_hit_virusesabundantfewer$taxname_lca_NTorNR)


In [ ]:
thresholded_hit_virusesabundantfewer

In [ ]:
## first remove more columns
## removing more
colnames(thresholded_hit_virusesabundantfewer)
thresholded_hit_virusesabundantfewcols <- thresholded_hit_virusesabundantfewer
## relocate()
thresholded_hit_virusesabundantfewcols <- thresholded_hit_virusesabundantfewcols %>% 
  relocate(tax_superkingdom_NTorNR, .before = gene_NTorNR) %>% relocate(tax_clade_NTorNR, .before = gene_NTorNR) %>% 
  relocate(tax_kingdom_NTorNR, .before = gene_NTorNR) %>% relocate(tax_phylum_NTorNR, .before = gene_NTorNR) %>% 
  relocate(tax_class_NTorNR, .before = gene_NTorNR) %>% relocate(tax_order_NTorNR, .before = gene_NTorNR) %>% 
  relocate(tax_family_NTorNR, .before = gene_NTorNR) %>% relocate(tax_genus_NTorNR, .before = gene_NTorNR) %>% 
  relocate(tax_species_NTorNR, .before = gene_NTorNR)

## first arrange by desc(abundance)
thresholded_hit_virusesabundantfewcols <- thresholded_hit_virusesabundantfewcols %>% arrange(desc(rel_abundance))


In [ ]:
thresholded_hit_virusesabundantfewcols

In [ ]:
## removing Bacteriophage sp. and LCA all viruses
thresholded_hit_virusesabundantfewcols <- thresholded_hit_virusesabundantfewcols %>% 
  dplyr::filter(taxname_lca_NTorNR != "Viruses")
thresholded_hit_virusesabundantfewcols <- thresholded_hit_virusesabundantfewcols %>% 
  dplyr::filter(taxname_lca_NTorNR != "Bacteriophage sp.")
n_distinct(thresholded_hit_virusesabundantfewcols$taxname_lca_NTorNR)

# quantile(thresholded_hit_virusesabundantfewcols$coverage)
# mean(thresholded_hit_virusesabundantfewcols$coverage)


In [ ]:
thresholded_hit_virusesabundantfewcols
n_distinct(thresholded_hit_virusesabundantfewcols$clusterLCA)
thresholded_hit_virusesabundantfewcols$clusterLCA[11]

In [ ]:
## now for separate metacoder tree of just fish-associated viruses:
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols %>% 
  dplyr::filter(clusterLCA == "Phoenicopteridae parvo-like hybrid virus" | 
                  clusterLCA == "Wuhan spiny eel influenza virus" | 
                  clusterLCA == "Siamese algae-eater influenza-like virus" |
                ### late Dec realized this list misses Betainfluenzavirus influenzae
                ## also Japanese_seabream_poxvirus - but this isn't a clusterLCA turns out not in Sept24 Salmon input at all
                ## in Feb adding 4 betanodavirus contigs
                  clusterLCA == "Betainfluenzavirus" | 
                  clusterLCA == "Betanodavirus epinepheli" | 
                  clusterLCA == "Betanodavirus" | 
                  clusterLCA == "Betainfluenzavirus influenzae" | 
                  clusterLCA == "Deltainfluenzavirus" | 
                  clusterLCA == "Cyprinid herpesvirus 1" | 
                  clusterLCA == "Cyvirus cyprinidallo3" | 
                  clusterLCA == "Cyprivirus A" | 
                  clusterLCA == "Zebrafish picornavirus 1" | 
                  clusterLCA == "Wuhan sharpbelly picornavirus 1" | 
                  clusterLCA == "Zebrafish picornavirus 2" | 
                  clusterLCA == "Coleura bat picornavirus" | 
                  clusterLCA == "Fish-associated picorna-like virus 3" | 
                  clusterLCA == "Barramundi calicivirus 1" | 
                  clusterLCA == "Wenzhou pacific spadenose shark paramyxovirus" | 
                  clusterLCA == "Wuhan japanese halfbeak arterivirus" |
                  clusterLCA == "Atlantic salmon calicivirus" |
                  clusterLCA == "Oscivirus" | 
                  clusterLCA == "Myotis ricketti associated fish calicivirus" | 
                  clusterLCA == "Salmovirus WFRC1" |
                  clusterLCA == "Wuhan japanese halfbeak arterivirus" | 
                  clusterLCA == "Flumine Astrovirus 3" |
                  clusterLCA == "Blotched snakehead virus" | 
                  clusterLCA == "Rocky Mountain birnavirus" | 
                  clusterLCA == "Snakehead retrovirus" | 
                  clusterLCA == "Wenzhou shark flavivirus" | 
                  clusterLCA == "Cutthroat trout virus" | 
                  clusterLCA == "Aquareovirus ctenopharyngodontis" | 
                  clusterLCA == "Guangdong catfish astro-like virus" | 
                  clusterLCA == "Ranavirus epinephelus1" | 
                  clusterLCA == "Carp edema virus" | 
                  clusterLCA == "Ranavirus" | 
                  clusterLCA == "African cichlid piscichuvirus" | 
                  clusterLCA == "Chuvivirus" | 
                  clusterLCA == "Hardyhead chuvirus" | 
                  clusterLCA == "Salarius guttatus piscichuvirus" | 
                  clusterLCA == "Sea turtle neural virus 1" | 
                  clusterLCA == "Fish-associated parvo-like hybrid virus" | 
                  clusterLCA == "Luscinia sibilans parvo-like hybrid virus" | 
                  clusterLCA == "Phoenicopterus roseus parvo-like hybrid virus" | 
                  clusterLCA == "Psittacidae aveparvovirus" | 
                  clusterLCA == "Psittaciform aveparvovirus" | 
                  clusterLCA == "Sprivivirus cyprinus" | 
                  clusterLCA == "Pimephales minnow adintovirus" | 
                  clusterLCA == "Astyanax tetra cavefish adintovirus" | 
                  clusterLCA == "Larimichthys croaker adintovirus" | 
                  clusterLCA == "Adintovirus anguilla1" | 
                  clusterLCA == "Catfish adomavirus" | 
                  clusterLCA == "Blueface angelfish adomavirus" | 
                  clusterLCA == "Leatherback sea turtle adomavirus" | 
                  clusterLCA == "Symphysodon discus adomavirus 1" | 
                  clusterLCA == "Isavirus" | 
                                ## in Feb also adding Isavirus_salaris
                  clusterLCA == "Isavirus salaris" | 
                  clusterLCA == "Salmon aquaparamyxovirus" | 
                  clusterLCA == "Wenling red spikefish hantavirus" | 
                  clusterLCA == "Murray-Darling rainbowfish hantavirus")
                  #                  clusterLCA == "Deltainfluenzavirus" | no Pimephales minnow adintovirus Astyanax tetra cavefish adintovirus Larimichthys croaker adintovirus Adintovirus anguilla1


In [ ]:
## later on code for figures
thresholded_hit_virusesabundantfewcols2_tohighlight


In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$taxname_lca_NTorNR)
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
## remove some few rows with taxnameLCA far from clusterLCA

#n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$taxname_lca_NTorNR)

thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  filter(!taxname_lca_NTorNR %in% c(
    "Anas platyrhynchos papillomavirus 2",
    "brine shrimp yue-like virus 3",
    "Caudoviricetes sp.",
    "Decapodiridovirus",
    "Fadolivirus algeromassiliense",
    "Gallid alphaherpesvirus 1",
    "Hubei orthoptera virus 4",
    "Iridoviridae",
    "Megrivirus",
    "Mimivirus sp.",
    "Narnavirus sp.",
    "Nucleocytoviricota",
    "Orthonairovirus",
    "Pandoravirus aubagnensis",
    "Riboviria sp.",
    "Severe acute respiratory syndrome coronavirus 2",
    "Alphabaculovirus spliturae",
    "Fadolivirus algeromassiliense",
    "Iltovirus gallidalpha1",
    "Monkeypox virus",
    "Phlebovirus limboense"
  ))
#taxname_lca_NTorNR
# Anas platyrhynchos papillomavirus 2
# brine shrimp yue-like virus 3
# Caudoviricetes sp.
# Decapodiridovirus
# Fadolivirus algeromassiliense
# Gallid alphaherpesvirus 1
# Hubei orthoptera virus 4
# Iridoviridae
# Megrivirus
# Mimivirus sp.
# Narnavirus sp.
# Nucleocytoviricota
# Orthonairovirus
# Pandoravirus aubagnensis
# Riboviria sp.
## Severe acute respiratory syndrome coronavirus 2

#Severe acute respiratory syndrome coronavirus 2
# Phlebovirus limboense
# brine shrimp yue-like virus 3
#Gallid alphaherpesvirus 1
#Narnavirus sp.
#Mimivirus sp.
#Alphabaculovirus spliturae
#Fadolivirus algeromassiliense
#Iltovirus gallidalpha1

n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$taxname_lca_NTorNR)
#unique(thresholded_hit_virusesabundantfewcols2_tohighlight$taxname_lca_NTorNR)


In [ ]:
## also remove handful of phage sequences that snuck in
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% dplyr::filter(viruscategorysimple_NTorNR != "Phage")
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$taxname_lca_NTorNR)


In [ ]:

thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA <- gsub("\\(", " ", thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)
thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA <- gsub("\\)", " ", thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
## this is very similar to fish-associated used in Fig1C notebook
# but that dataframe then has all sprivivirus replaced with 2 reference genomes

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight

In [ ]:
#then use the key that gives the cluster_LCA_curated - make sure to rename 0x and 1y names
# # Load the curation key
####### curation_key <- read_tsv("curated_virus_contigs_for_modeling_dec.tsv")
#we will want to use the OG clusterLCA names for metacoder trees though, without using our curated names!
#for actual identity plot though, use the cluster_LCA_curated


## once all of this is done - return to notebook for fig1C and redo that but keeping all metadata not just tissue...


In [ ]:
## feb 2026 adding 4 betanodavirus sequences to curated tsv
thresholded_hit_virusesabundantfewcols2_tohighlight %>% dplyr::filter(clusterLCA == "Betanodavirus epinepheli" | clusterLCA == "Betanodavirus")

In [ ]:
## standardize betanodavirus by changing all clusterLCA to betanodavirus
## Betanodavirus_epinepheli to Betanodavirus
## and
## Betanodavirus epinepheli to Betanodavirus
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(across(where(is.character), ~str_replace_all(.x, c(
    "Betanodavirus_epinepheli" = "Betanodavirus",
    "Betanodavirus epinepheli" = "Betanodavirus"
  ))))

n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)
## finally Betanodavirus to Redspotted grouper nervous necrosis virus only in clusterLCA
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% 
  mutate(clusterLCA = str_replace_all(clusterLCA, c("Betanodavirus" = "Redspotted grouper nervous necrosis virus")))

In [ ]:
## similarly combine Isavirus & Isavirus_salaris

thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(across(where(is.character), ~str_replace_all(.x, c(
    "Isavirus_salaris" = "Isavirus",
    "Isavirus salaris" = "Isavirus"
  ))))

n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight

In [ ]:
## update with 4 nodaviruses (changing names later) curated_virus_contigs_for_modeling_feb26
## more Isavirus in mar26

## stay in this wd...

## feb26 version has all of code below to fix names - these are updated in mar26 version!!
#curation_key <- read_tsv("curated_virus_contigs_for_modeling_feb26.tsv")
curation_key <- read_tsv("curated_virus_contigs_for_modeling_mar26.tsv")

## but actually use as true key just for unique values of cluster_LCA!
#
curation_key0 <- curation_key %>% select(clusterLCA_curated,clusterLCA)
head(curation_key0)

In [ ]:
## adding because strange duplication of sprivivirus entries
# curation_key <- curation_key %>%
#   mutate(
#     clusterLCA_curated = clusterLCA_curated %>%
#       str_replace("Spring viraemia of carp virus", "Sprivivirus cyprinus")
#   )


head(curation_key)
## also Nodavirus fix
curation_key <- curation_key %>%
  mutate(across(where(is.character), ~str_replace_all(.x, c(
    "Betanodavirus_epinepheli" = "Betanodavirus"
  ))))


In [ ]:
unique(curation_key$clusterLCA_curated)
n_distinct(curation_key$clusterLCA_curated)
n_distinct(curation_key$clusterLCA)

In [ ]:
## this is a problem - we do not want to group by clusterLCA but clusterLCA_curated
## also ungroup after this!
#curation_key <- curation_key0 %>% group_by(clusterLCA) %>% summarize(clusterLCA_curated = first(clusterLCA_curated), clusterLCA = first(clusterLCA))
#curation_key <- curation_key0 %>% ungroup()


# curation_key <- curation_key0 %>% group_by(clusterLCA_curated) %>% summarize(clusterLCA_curated = first(clusterLCA_curated), clusterLCA = first(clusterLCA))
# curation_key
## hmm - we need to more carefully merge based on all contig names not this...
## do not use group_by at all!

In [ ]:
##also indicate which of these are “endogenous or likely not fish-related (endogenous_xxx) and “insufficient evidence” (insufficient_YY)
## renaming sets - was 0... & 1... change "0..." to "endogenous_" & "1..." to "insufficient_"
##
## then rename columns like so:

curation_key <- curation_key %>%
  mutate(
    clusterLCA_curated = clusterLCA_curated %>%
      str_replace("^0", "endogenous_or_nonfish ") %>%
      str_replace("^1", "insufficient_evidence ")
  )

cat("Updated clusterLCA_curated column\n")

# Show examples of renamed values
cat("\nExamples with 'endogenous_or_nonfish':\n")
print(head(curation_key %>% filter(str_detect(clusterLCA_curated, "^endogenous_or_nonfish")), 5))

cat("\nExamples with 'insufficient_evidence':\n")
print(head(curation_key %>% filter(str_detect(clusterLCA_curated, "^insufficient_evidence")), 5))

In [ ]:
head(curation_key)

In [ ]:
## also 
curation_key <- curation_key %>%
  mutate(
    clusterLCA = clusterLCA %>%
      str_replace("anguilla1", "anguilla10665")
  )
head(curation_key)

In [ ]:
curation_key
## hmm - we need to more carefully merge based on all contig names not this...

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    clusterLCA = clusterLCA %>%
      str_replace("anguilla1", "anguilla10665")
  )
head(thresholded_hit_virusesabundantfewcols2_tohighlight)

In [ ]:
## similarly fix cluster in poxvirus

## similarly fix cluster in poxvirus
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    contig_withLCA_withcluster = str_replace(
      contig_withLCA_withcluster,  # specify the column name
      "(CLUSTER20_Carp_edema_virus\\|)75(\\|44)",
      "\\174\\2"
    )
  )

# Check what changed
thresholded_hit_virusesabundantfewcols2_tohighlight %>% 
  filter(str_detect(contig_withLCA_withcluster, "R20_Carp_edema_virus.*74\\|44")) %>%
  select(contig_withLCA_withcluster)


In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight

In [ ]:
n_distinct(curation_key$clusterLCA)
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
## isavirus check

#thresholded_hit_virusesabundantfewcols2_tohighlight %>% dplyr::filter(grepl('Isavirus', clusterLCA))


In [ ]:
#curation_key %>% dplyr::filter(grepl('Isavirus', clusterLCA))


In [ ]:
#curation_key %>% dplyr::filter(grepl('Spri', clusterLCA))

## adding because strange duplication of sprivivirus entries
# curation_key <- curation_key %>%
#   mutate(
#     clusterLCA_curated = clusterLCA_curated %>%
#       str_replace("Spring viraemia of carp virus", "Sprivivirus cyprinus")
#   )


In [ ]:
## need to run this command on curation_key as well!
curation_key <- curation_key %>%
  mutate(across(where(is.character), ~str_replace_all(.x, c(
    "Isavirus_salaris" = "Isavirus",
    "Isavirus salaris" = "Isavirus"
  ))))

#n_distinct(curation_key$clusterLCA)


In [ ]:
# Get unique values from each dataframe
unique_curation <- unique(curation_key$clusterLCA) %>% sort()
unique_contigs <- unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA) %>% sort()

# Count distinct values
cat("Distinct values in curation_key:", length(unique_curation), "\n")
cat("Distinct values in thresholded dataframe:", length(unique_contigs), "\n")

# Check if they're identical
if (identical(unique_curation, unique_contigs)) {
  cat("\n✓ The unique clusterLCA values are identical!\n")
} else {
  cat("\n✗ The unique clusterLCA values are NOT identical\n")
  
  # Find differences
  only_in_key <- setdiff(unique_curation, unique_contigs)
  only_in_contigs <- setdiff(unique_contigs, unique_curation)
  
  cat("\nValues only in curation_key:", length(only_in_key), "\n")
  if (length(only_in_key) > 0) {
    cat("First 10:\n")
    print(head(only_in_key, 10))
  }
  
  cat("\nValues only in contigs:", length(only_in_contigs), "\n")
  if (length(only_in_contigs) > 0) {
    cat("First 10:\n")
    print(head(only_in_contigs, 10))
  } 
    }

In [ ]:
## now we can join then make metacoder tree and identity plots using clusterLCA and then clusterLCA_curated respectively

In [ ]:
## BUT - WE WANT TO JOIN BY contig_withLCA_withcluster NOT clusterLCA


In [ ]:
curation_key2 <- curation_key %>% select(-clusterLCA)
head(curation_key2)

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight0 <- thresholded_hit_virusesabundantfewcols2_tohighlight


In [ ]:
#thresholded_hit_virusesabundantfewcols2_tohighlight <- left_join(thresholded_hit_virusesabundantfewcols2_tohighlight,curation_key)
thresholded_hit_virusesabundantfewcols2_tohighlight <- left_join(thresholded_hit_virusesabundantfewcols2_tohighlight,curation_key2)

In [ ]:
head(thresholded_hit_virusesabundantfewcols2_tohighlight0)
head(thresholded_hit_virusesabundantfewcols2_tohighlight)

In [ ]:
## now check to see how many NAs in clusterLCA_curated & what are their unique values of clusterLCA

# Find rows with NA in clusterLCA_curated
na_rows <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  filter(is.na(clusterLCA_curated))

# Count and show unique clusterLCA values
cat("Number of rows with NA in clusterLCA_curated:", nrow(na_rows), "\n\n")

if (nrow(na_rows) > 0) {
  unique_clusterLCA <- unique(na_rows$clusterLCA)
  
  cat("Number of unique clusterLCA values:", length(unique_clusterLCA), "\n\n")
  
  cat("Unique clusterLCA values for NAs:\n")
  print(unique_clusterLCA)
  
  # Optional: show counts per clusterLCA
  cat("\nCounts per clusterLCA:\n")
  na_summary <- na_rows %>%
    count(clusterLCA, sort = TRUE)
  print(na_summary)
# Save to file for review
#  write_tsv(na_rows, "na_clusterLCA_detailed.tsv")
#  cat("\nSaved summary to na_clusterLCA_detailed.tsv\n")
} else {
  cat("No NAs found in clusterLCA_curated!\n")

    }

In [ ]:
## now that we've confirmed only Sprivi are missing NA let's get those clusterLCA_curated values filled in using coalesce


# Create a before/after comparison
before_after <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    was_na = is.na(clusterLCA_curated),
    clusterLCA_curated_new = coalesce(clusterLCA_curated, clusterLCA)
  )

# Summary of replacements
replacement_summary <- before_after %>%
  filter(was_na) %>%
  count(clusterLCA, name = "n_rows_replaced") %>%
  arrange(desc(n_rows_replaced))

cat("Summary of replacements:\n")
print(replacement_summary)
cat("\nTotal rows where NA will be replaced:", sum(replacement_summary$n_rows_replaced), "\n\n")



In [ ]:
# Apply the change
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    clusterLCA_curated = coalesce(clusterLCA_curated, clusterLCA)
  )

# Verify
cat("Verification - NAs remaining:", 
    sum(is.na(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)), "\n")


In [ ]:
## that somehow lost our sprivi name change
thresholded_hit_virusesabundantfewcols2_tohighlight %>% dplyr::filter(grepl('Spri', clusterLCA_curated))


In [ ]:
## adding because strange duplication of sprivivirus entries
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    clusterLCA_curated = clusterLCA_curated %>%
      str_replace("Sprivivirus cyprinus", "Spring viraemia of carp virus")
  )

In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)

In [ ]:
## also add a category column: Novel, Insufficient Evidence, Endogenous or not fish-associated, or Known

# thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
#   mutate(
#     Category = case_when(
#       str_detect(clusterLCA_curated, regex("Novel", ignore_case = TRUE)) ~ "Novel",
#       str_detect(clusterLCA_curated, regex("insufficient_evidence", ignore_case = TRUE)) ~ "Insufficient Evidence",
#       str_detect(clusterLCA_curated, regex("endogenous_or_nonfish", ignore_case = TRUE)) ~ "Endogenous or not fish-associated",
#       TRUE ~ "Known"
#     )
#   )



thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    Category = case_when(
      clusterLCA_curated %in% c(
        "Zebrafish adomavirus",
        "Zebrafish rhabdo-like virus",
        "Zebrafish gut calicivirus",
        "Zebrafish neurinoma calicivirus",
        "Zebrafish systemic calicivirus",
        "Zebrafish gonadal hantavirus",
        "Zebrafish cardiovascular hantavirus",
        "Zebrafish neural hantavirus",
        "Zebrafish arterivirus",
        "Influenza Z virus",
        "Zebrafish influenza-like virus",
        "Zebrafish systemic paramyxovirus",
        "Zebrafish hepatic paramyxovirus",
        "Danio blood picornavirus/Zebrafish picornavirus 2",
        "Zebrafish jaw picornavirus",
        "Zebrafish jaw poxvirus"
      ) ~ "Novel",
      str_detect(clusterLCA_curated, regex("insufficient_evidence", ignore_case = TRUE)) ~ "Insufficient Evidence",
      str_detect(clusterLCA_curated, regex("endogenous_or_nonfish", ignore_case = TRUE)) ~ "Endogenous or not fish-associated",
      TRUE ~ "Known"
    )
  )



In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)


In [ ]:
## now update the table with new names, also ZFPV2 to novel
## later below change the order

## also make subset filtering out the insufficient_evidence & endogenous_or_nonfish

In [ ]:
# replacement mapping - no longer needed


In [ ]:
## save
#write.table(thresholded_hit_virusesabundantfewcols2_tohighlight, file = paste0("taxonomy_hits_viruses_clusters_nosequencenotargetsforsalmon_fishassociated_sept17_feb26update.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [ ]:

## Now metacoder code, first cleaning up then actual Metacoder loop:
colnames(thresholded_hit_virusesabundantfewcols2_tohighlight)



In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight %>% dplyr::filter(clusterLCA_curated == "Redspotted grouper nervous necrosis virus")

In [ ]:
#thresholded_hit_virusesabundantfewcols2_tohighlight$bioprojectsbyLCAcluster <- NULL
#thresholded_hit_virusesabundantfewcols2_tohighlight$contigsbyLCAcluster <- NULL
thresholded_hit_virusesabundantfewcols2_tohighlight

In [ ]:
## get 2 new metrics: bioprojectsbyLCAcluster & contigsbyLCAcluster
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% group_by(clusterLCA_curated) %>% mutate(bioprojectsbyLCAcluster=n_distinct(bioproject)) %>%
  ungroup()

thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% group_by(clusterLCA_curated) %>%
  mutate(contigsbyLCAcluster = n()) %>%
  ungroup()

thresholded_hit_virusesabundantfewcols2_tohighlight


In [ ]:
## then at bottom is code for plot...

In [ ]:
## Now Metacoder loop:


In [ ]:
colnames(thresholded_hit_virusesabundantfewcols2_tohighlight)

## keeping pident_NTorNR
thresholded_hit_virusesabundantfewcols100 <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% select(-node) %>% select(-coverage) %>% select(-taxid_NTorNR) %>% select(-evalue_NTorNR) %>% select(-gene_NTorNR) %>% select(-allele_NTorNR) %>% select(-sumperc_cov_NTorNR) %>% select(-alnlen_NTorNR) %>% select(-mismatch_NTorNR) %>% select(-qcov_NTorNR) %>% select(-target_title_NTorNR) %>% select(-sumalnlen_NTorNR) %>% select(-bits_percmax_NTorNR) %>% select(-target_NTorNR)
colnames(thresholded_hit_virusesabundantfewcols100)

thresholded_hit_virusesabundantfewcols101 <- thresholded_hit_virusesabundantfewcols100
library(taxize)


In [ ]:
## actual metacoder loop
inputs <- list(
  "thresholded_hit_virusesabundantfewcols100",
  "thresholded_hit_virusesabundantfewcols101"
)

max_retries <- 3  # Maximum retries per input
total_attempts <- 0  # Track total attempts
success <- FALSE  # Flag to indicate success

for (input in inputs) {
  retry_count <- 0  # Reset retry counter for each input
  
  while (!success && retry_count < max_retries) {
    total_attempts <- total_attempts + 1  # Increment total attempts
    retry_count <- retry_count + 1  # Increment retry count for the current input
    
    tryCatch(
      {
        message(paste("Attempt", total_attempts, "with input:", input, "(Retry", retry_count, "of", max_retries, ")..."))
        
        # Attempt the command with the current input
        obj_viraltopabundance3o <- lookup_tax_data(
          get(input),  # Use the current input
          type = "taxon_name",
          column = "clusterLCA",
          ask = FALSE
        )
        
        # If successful, set success to TRUE and break out of the loop
        success <- TRUE
        message("lookup_tax_data executed successfully.")
      },
      error = function(e) {
        # Handle errors
        message(paste("Attempt", total_attempts, "failed with input:", input, "."))
        
        if (retry_count < max_retries) {
          message("Retrying in 8 seconds...")
          Sys.sleep(8)  # Wait before retrying
        } else {
          message(paste("All", max_retries, "retries failed for input:", input, ". Moving to the next input..."))
        }
      }
    )
  }
  
  # If successful, break out of the for-loop
  if (success) break
}

# If all inputs failed
if (!success) {
  stop("Failed after 15 total attempts across all inputs.")
}




In [ ]:
obj_viraltopabundance3o

In [ ]:
## save right away!
obj_viraltopabundance3 <- obj_viraltopabundance3o
save(obj_viraltopabundance3o, file = "obj_taxonomy_hits_viruses_fishspecificMar26.Rdata")

obj_viraltopabundance3$data <- calc_taxon_abund(obj_viraltopabundance3, "query_data")
save(obj_viraltopabundance3, file = "obj_taxonomy_hits_viruses_fishspecificwithlengthandbitscoreMar26t.Rdata")


In [ ]:
## BEST AT CAPTURING ALL LABELS, NOTE BACKGROUND LIGHT GRAY
## first relabundance tree
set.seed(11) # This makes the plot appear the same each time it is run 
metatree32s2 <- heat_tree(obj_viraltopabundance3, 
                          node_label = taxon_names,
                          node_size = rel_abundance,  ## works after line above, note change next to rel_abundance
                          node_color = rel_abundance, 
                          node_size_axis_label = "Relative abundance",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundance_Feb_",Sys.Date(),".png", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundance_Feb_",Sys.Date(),".pdf", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
## also by contigsbyLCAcluster & bioprojectsbyLCAcluster

set.seed(11) # This makes the plot appear the same each time it is run 
metatree32s2 <- heat_tree(obj_viraltopabundance3, 
                          node_label = taxon_names,
                          node_size = contigsbyLCAcluster,  ## works after line above, note change next to rel_abundance
                          node_color = contigsbyLCAcluster, 
                          node_size_axis_label = "Transcripts by LCA cluster",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contigsbyLCAcluster_Feb_",Sys.Date(),".png", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contigsbyLCAcluster_Feb_",Sys.Date(),".pdf", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)

metatree32s2 <- heat_tree(obj_viraltopabundance3, 
                          node_label = taxon_names,
                          node_size = bioprojectsbyLCAcluster,  ## works after line above, note change next to rel_abundance
                          node_color = bioprojectsbyLCAcluster, 
                          node_size_axis_label = "Bioprojects by LCA cluster",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bioprojectsbyLCAcluster_Feb_",Sys.Date(),".png", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bioprojectsbyLCAcluster_Feb_",Sys.Date(),".pdf", sep=""), metatree32s2, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
## then bitscore tree
set.seed(11) # This makes the plot appear the same each time it is run 
## MAKE A VERSION WITHOUT LEGEND...also making bigger for fish-specific labels!
metatree322 <- heat_tree(obj_viraltopabundance3, 
                         node_label = taxon_names,
                         node_size = bits_NTorNR,  ## works after line above, note change from rel_abundance
                         node_color = bits_NTorNR, 
                         node_size_axis_label = "Bitscore",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
 #                         node_size_interval = c(1e1,1e7),
 #                         node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations ALSO NOTE NEW DIMENSIONS
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscore_Feb_",Sys.Date(),".png", sep=""), metatree322, width = 28, height = 25, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscore_Feb_",Sys.Date(),".pdf", sep=""), metatree322, width = 28, height = 25, units = "in", limitsize = FALSE)


In [ ]:
## then bitscore tree without labels
set.seed(11) # This makes the plot appear the same each time it is run 
## MAKE A VERSION WITHOUT LEGEND...also making bigger for fish-specific labels!
metatree322 <- heat_tree(obj_viraltopabundance3, 
                         node_label = taxon_names,
                         node_size = bits_NTorNR,  ## works after line above, note change from rel_abundance
                         node_color = bits_NTorNR, 
                         node_size_axis_label = "Bitscore",
                          node_label_size_range = c(0.005, 0.015),
                          make_node_legend = FALSE,
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations ALSO NOTE NEW DIMENSIONS
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscorenolabel_Feb_",Sys.Date(),".png", sep=""), metatree322, width = 28, height = 25, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscorenolabel_Feb_",Sys.Date(),".pdf", sep=""), metatree322, width = 28, height = 25, units = "in", limitsize = FALSE)


In [ ]:
## then contig length tree
metatree323 <- heat_tree(obj_viraltopabundance3, 
                         node_label = taxon_names,
                         node_size = length,  ## works after line above, note change from rel_abundance
                         node_color = length, 
                         node_size_axis_label = "Contig length",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contiglength_Feb_",Sys.Date(),".png", sep=""), metatree323, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contiglength_Feb_",Sys.Date(),".pdf", sep=""), metatree323, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
## NOW REPEAT BUT KEEPING ONLY NOVEL & KNOWN...Category
thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  dplyr::filter(Category == "Novel" | Category == "Known")
thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown

In [ ]:

## keeping pident_NTorNR
thresholded_hit_virusesabundantfewcols10 <- thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown %>% select(-node) %>% select(-coverage) %>% select(-taxid_NTorNR) %>% select(-evalue_NTorNR) %>% select(-gene_NTorNR) %>% select(-allele_NTorNR) %>% select(-sumperc_cov_NTorNR) %>% select(-alnlen_NTorNR) %>% select(-mismatch_NTorNR) %>% select(-qcov_NTorNR) %>% select(-target_title_NTorNR) %>% select(-sumalnlen_NTorNR) %>% select(-bits_percmax_NTorNR) %>% select(-target_NTorNR)
colnames(thresholded_hit_virusesabundantfewcols10)

thresholded_hit_virusesabundantfewcols11 <- thresholded_hit_virusesabundantfewcols10
library(taxize)


In [ ]:
## actual metacoder loop
inputs <- list(
  "thresholded_hit_virusesabundantfewcols10",
  "thresholded_hit_virusesabundantfewcols11"
)

max_retries <- 3  # Maximum retries per input
total_attempts <- 0  # Track total attempts
success <- FALSE  # Flag to indicate success

for (input in inputs) {
  retry_count <- 0  # Reset retry counter for each input
  
  while (!success && retry_count < max_retries) {
    total_attempts <- total_attempts + 1  # Increment total attempts
    retry_count <- retry_count + 1  # Increment retry count for the current input
    
    tryCatch(
      {
        message(paste("Attempt", total_attempts, "with input:", input, "(Retry", retry_count, "of", max_retries, ")..."))
        
        # Attempt the command with the current input
        obj_viraltopabundance2o <- lookup_tax_data(
          get(input),  # Use the current input
          type = "taxon_name",
          column = "clusterLCA",
          ask = FALSE
        )
        
        # If successful, set success to TRUE and break out of the loop
        success <- TRUE
        message("lookup_tax_data executed successfully.")
      },
      error = function(e) {
        # Handle errors
        message(paste("Attempt", total_attempts, "failed with input:", input, "."))
        
        if (retry_count < max_retries) {
          message("Retrying in 8 seconds...")
          Sys.sleep(8)  # Wait before retrying
        } else {
          message(paste("All", max_retries, "retries failed for input:", input, ". Moving to the next input..."))
        }
      }
    )
  }
  
  # If successful, break out of the for-loop
  if (success) break
}

# If all inputs failed
if (!success) {
  stop("Failed after 15 total attempts across all inputs.")
}



In [ ]:
## save right away!
obj_viraltopabundance2 <- obj_viraltopabundance2o
save(obj_viraltopabundance2o, file = "obj_taxonomy_hits_viruses_fishspecific_novelknownMar26.Rdata")

obj_viraltopabundance2$data <- calc_taxon_abund(obj_viraltopabundance2, "query_data")
save(obj_viraltopabundance2, file = "obj_taxonomy_hits_viruses_fishspecificwithlengthandbitscore_novelknownMar26t.Rdata")


In [ ]:
## BEST AT CAPTURING ALL LABELS, NOTE BACKGROUND LIGHT GRAY
## first relabundance tree
set.seed(11) # This makes the plot appear the same each time it is run 
metatree22s2 <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = rel_abundance,  ## works after line above, note change next to rel_abundance
                          node_color = rel_abundance, 
                          node_size_axis_label = "Relative abundance",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundance_novelknown_Feb_",Sys.Date(),".png", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundance_novelknown_Feb_",Sys.Date(),".pdf", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)


## also by contigsbyLCAcluster & bioprojectsbyLCAcluster

set.seed(11) # This makes the plot appear the same each time it is run 
metatree22s3 <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = contigsbyLCAcluster,  ## works after line above, note change next to rel_abundance
                          node_color = contigsbyLCAcluster, 
                          node_size_axis_label = "Transcripts by LCA cluster",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contigsbyLCAcluster_novelknown_Feb_",Sys.Date(),".png", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contigsbyLCAcluster_novelknown_Feb_",Sys.Date(),".pdf", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)

metatree22s4 <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = bioprojectsbyLCAcluster,  ## works after line above, note change next to rel_abundance
                          node_color = bioprojectsbyLCAcluster, 
                          node_size_axis_label = "Bioprojects by LCA cluster",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bioprojectsbyLCAcluster_novelknown_Feb_",Sys.Date(),".png", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bioprojectsbyLCAcluster_novelknown_Feb_",Sys.Date(),".pdf", sep=""), metatree22s2, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
## then bitscore tree
set.seed(11) # This makes the plot appear the same each time it is run 
## MAKE A VERSION WITHOUT LEGEND...also making bigger for fish-specific labels!
metatree222 <- heat_tree(obj_viraltopabundance2, 
                         node_label = taxon_names,
                         node_size = bits_NTorNR,  ## works after line above, note change from rel_abundance
                         node_color = bits_NTorNR, 
                         node_size_axis_label = "Bitscore",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                         node_size_interval = c(1e1,1e7),
#                         node_color_interval = c(1e1,1e7),
#                         background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations ALSO NOTE NEW DIMENSIONS
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscore_novelknown_Mar_",Sys.Date(),".png", sep=""), metatree222, width = 28, height = 25, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscore_novelknown_Mar_",Sys.Date(),".pdf", sep=""), metatree222, width = 28, height = 25, units = "in", limitsize = FALSE)

## then bitscore tree without labels
set.seed(11) # This makes the plot appear the same each time it is run 
## MAKE A VERSION WITHOUT LEGEND...also making bigger for fish-specific labels!
metatree222 <- heat_tree(obj_viraltopabundance2, 
                         node_label = taxon_names,
                         node_size = bits_NTorNR,  ## works after line above, note change from rel_abundance
                         node_color = bits_NTorNR, 
                         node_size_axis_label = "Bitscore",
                          node_label_size_range = c(0.005, 0.015),
                          make_node_legend = FALSE,
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
#                          background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations ALSO NOTE NEW DIMENSIONS
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscorenolabel_novelknown_Mar_",Sys.Date(),".png", sep=""), metatree222, width = 28, height = 25, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_bitscorenolabel_novelknown_Mar_",Sys.Date(),".pdf", sep=""), metatree222, width = 28, height = 25, units = "in", limitsize = FALSE)

## then contig length tree
metatree223 <- heat_tree(obj_viraltopabundance2, 
                         node_label = taxon_names,
                         node_size = length,  ## works after line above, note change from rel_abundance
                         node_color = length, 
                         node_size_axis_label = "Contig length",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
#                          background_color = "gray95",
                         # node_color_axis_label = "Samples with reads",
                         layout = "davidson-harel", # The primary layout algorithm
                         initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations
#ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contiglength_novelknown_Feb_",Sys.Date(),".png", sep=""), metatree223, width = 44, height = 34, units = "in", limitsize = FALSE)
#ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_contiglength_novelknown_Feb_",Sys.Date(),".pdf", sep=""), metatree223, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
set.seed(11) # This makes the plot appear the same each time it is run 
metatree22s2f <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = rel_abundance,  ## works after line above, note change next to rel_abundance
                          node_color = rel_abundance, 
                          node_size_axis_label = "Abundance",
                          node_label_size_range = c(0.005, 0.015),
                          make_node_legend = FALSE,
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
#                          node_size_interval = c(1e1,1e7),
#                          node_color_interval = c(1e1,1e7),
    #                      background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancenolabel_novelknown_Mar_",Sys.Date(),".png", sep=""), metatree22s2f, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancenolabel_novelknown_Mar_",Sys.Date(),".pdf", sep=""), metatree22s2f, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
set.seed(11) # This makes the plot appear the same each time it is run 
metatree22s2s <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = rel_abundance,  ## works after line above, note change next to rel_abundance
                          node_color = rel_abundance, 
                          node_size_axis_label = "Abundance",
                          node_label_size_range = c(0.005, 0.015),
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
                          node_size_interval = c(1e1,1e7),
                          node_color_interval = c(1e1,1e7),
#                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancescaledmax_novelknown_Mar_",Sys.Date(),".png", sep=""), metatree22s2s, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancescaledmax_novelknown_Mar_",Sys.Date(),".pdf", sep=""), metatree22s2s, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
set.seed(11) # This makes the plot appear the same each time it is run 
metatree22s2sf <- heat_tree(obj_viraltopabundance2, 
                          node_label = taxon_names,
                          node_size = rel_abundance,  ## works after line above, note change next to rel_abundance
                          node_color = rel_abundance, 
                          node_size_axis_label = "Abundance",
                          node_label_size_range = c(0.005, 0.015),
                          make_node_legend = FALSE,
                          node_label_max = 800,
                          edge_label_max = 500,
                          tree_label_max = 800,
                          node_color_digits = 0,
                          node_size_digits = 0,
                          edge_color_digits = 0,
                          edge_size_digits = 0,
                          repel_force = 1.5,
                          node_size_interval = c(1e1,1e7),
                          node_color_interval = c(1e1,1e7),
#                          background_color = "gray95",
                          layout = "davidson-harel", # The primary layout algorithm
                          initial_layout = "reingold-tilford") # The layout algorithm that initializes node locations

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancenolabelscaledmax_novelknown_Mar_",Sys.Date(),".png", sep=""), metatree22s2sf, width = 44, height = 34, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_metacodertree_relabundancenolabelscaledmax_novelknown_Mar_",Sys.Date(),".pdf", sep=""), metatree22s2sf, width = 44, height = 34, units = "in", limitsize = FALSE)


In [ ]:
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA)


In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)


In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight
## move zfpv2 from known to novel...

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(Category = if_else(
    clusterLCA_curated == "Danio blood picornavirus/Zebrafish picornavirus 2",
    "Novel",
    Category
  ))

thresholded_hit_virusesabundantfewcols2_tohighlight

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  mutate(
    Category = factor(Category, levels = c(
      "Novel",
      "Known",
      "Endogenous or not fish-associated",
      "Insufficient Evidence"
    ))
  )

unique(thresholded_hit_virusesabundantfewcols2_tohighlight$clusterLCA_curated)


In [ ]:
#write.table(thresholded_hit_virusesabundantfewcols2_tohighlight_names, file = paste0("RNaquarium_fishvirusstats_Sept_",Sys.Date(),".tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [ ]:
#write.table(thresholded_hit_virusesabundantfewcols2_tohighlight, file = paste0("taxonomy_hits_viruses_clusters_nosequencenotargetsforsalmon_sprivireplacediwth2genomes_fishassociated0_2025-10-09.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)

#taxonomy_hits_viruses_withclusters_nosequencenotargetsforsalmon_sprivireplacediwth2genomesfishassociated_2025-09-17.tsv taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_mostrecent.tsv
#write.table(thresholded_hit_virusesabundantfewcols2_tohighlight, file = paste0("RNaquarium_fishvirusstats_Septchk0_",Sys.Date(),".tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [ ]:
## plotting code:

# Step 1: Modify the tax_family_NTorNR column
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% replace_na(list(tax_family_NTorNR = "(Adomaviruses)"))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Salmovirus WFRC1",
                                     "(Rhabdo-like)", tax_family_NTorNR))
## repeat for more:
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Guangdong catfish astro-like virus",
                                     "(Astro-like)", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Fish-associated picorna-like virus 3",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Astyanax tetra cavefish adintovirus",
                                     "Adintoviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Salmovirus WFRC1",
                                     "(Rhabdo-like)", tax_family_NTorNR))
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Guangdong catfish astro-like virus",
                                     "(Astro-like)", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Fish-associated picorna-like virus 3",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Astyanax tetra cavefish adintovirus",
                                     "Adintoviridae", tax_family_NTorNR))

## a few more edge cases
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Coleura bat picornavirus",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Zebrafish picornavirus 2",
                                     "Picornaviridae", tax_family_NTorNR))

# thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
#   mutate(tax_family_NTorNR = if_else(clusterLCA == "Carp edema virus",
#                                      "Poxviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Fish-associated parvo-like hybrid virus",
                                     "Parvoviridae", tax_family_NTorNR))

#Pimephales minnow adintovirus
#Sprivivirus cyprinus Rhabdoviridae

## to Parvo-like
# Fish-associated parvo-like hybrid virus
# Fish-associated parvo-like hybrid virus

## check order
thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR[1]
unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR)




In [ ]:


thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>% 
  ungroup() %>% 
  mutate(has_parentheses = str_detect(tax_family_NTorNR, "\\(")) %>%
  arrange(Category, has_parentheses, tax_family_NTorNR, clusterLCA_curated) %>%
  select(-has_parentheses)

unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR)


In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$clusterLCA_curated)

In [ ]:
## NOW FACTOR 
# # Set taxname_lca_NTorNR as a factor with explicit levels in correct order
# thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
#   mutate(clusterLCA_curated = factor(clusterLCA_curated, levels = unique(clusterLCA_curated)))

# thresholded_hit_virusesabundantfewcols2_tohighlightforplot$clusterLCA_curated[1]
# # [1] Cyprinid herpesvirus 1
# # 18 Levels: Cyprinid herpesvirus 1 Aquabirnavirus Beihai conger calicivirus Hardyhead chuvirus ... Fish-associated picorna-like virus 3
# # >

# ## BUT WE WANT REVERSE?
# thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
#   mutate(clusterLCA_curated = factor(clusterLCA_curated, levels = rev(unique(clusterLCA_curated))))



## MOVING INTENTIONALLY ADMINISTERED KNOWN VIRUSES TO END:

thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(
    sort_order = case_when(
      clusterLCA_curated == "Cyprinid herpesvirus 3" ~ 1,
      clusterLCA_curated == "Redspotted grouper nervous necrosis virus" ~ 2,
      clusterLCA_curated == "Sprivivirus cyprinus" ~ 3,
      TRUE ~ 0
    )
  ) %>%
  mutate(has_parentheses = str_detect(tax_family_NTorNR, "\\(")) %>%
  arrange(Category, sort_order, has_parentheses, tax_family_NTorNR, clusterLCA_curated) %>%
  select(-sort_order) %>%
  select(-has_parentheses) %>%
  mutate(clusterLCA_curated = factor(clusterLCA_curated, levels = rev(unique(clusterLCA_curated))))

thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$clusterLCA_curated[1]

## rename NTclustered to just NT
thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev <- thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev %>%
  mutate(analysis_used = if_else(analysis_used == "NTclustered",
                                     "NT", analysis_used))


In [ ]:
## actual plot
# Step 4: Plot
## first small for visualizing
ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
                     aes(x = pident_NTorNR, y = clusterLCA_curated, color = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
#  scale_color_manual(values = c("NR" = "red", "NTclustered" = "blue")) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )


In [ ]:
# Check for NAs in the factor conversion
plot_data <- thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev %>%
  mutate(
    family_cluster = paste0(tax_family_NTorNR, " | ", clusterLCA_curated)
  )

# Check if any family_cluster values don't match when we try to create the factor
# First, let's see what levels we're creating
expected_levels <- paste0(
  levels(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$clusterLCA_curated) %>% 
    map_chr(~unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$tax_family_NTorNR[
      thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$clusterLCA_curated == .x])[1]),
  " | ",
  levels(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$clusterLCA_curated)
)

cat("Expected levels (first 5):\n")
print(head(expected_levels, 5))

# Check which family_cluster values don't match the expected levels
unmatched <- setdiff(unique(plot_data$family_cluster), expected_levels)

if (length(unmatched) > 0) {
  cat("\nFamily_cluster values that don't match expected levels:\n")
  print(unmatched)
  
  # Find which rows have these unmatched values
  problematic_rows <- plot_data %>%
    filter(family_cluster %in% unmatched) %>%
    select(clusterLCA_curated, tax_family_NTorNR, family_cluster)
  
  cat("\nProblematic rows:\n")
  print(problematic_rows)
}

# Check if there are multiple families per cluster
multiple_families <- thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev %>%
  group_by(clusterLCA_curated) %>%
  summarize(n_families = n_distinct(tax_family_NTorNR), .groups = "drop") %>%
  filter(n_families > 1)

if (nrow(multiple_families) > 0) {
  cat("\nClusters with multiple families:\n")
  print(multiple_families)
}

In [ ]:
## instead make table with family & cluster



In [ ]:
bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray70", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )


smalldotplotgray <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray70", "NT" = "gray20")) +
#  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )


smalldotplotbw <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray70", "NT" = "gray20")) +
#  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_bw(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )

smalldotplotlight <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray70", "NT" = "gray20")) +
#  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_light(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )

smalldotplotminimal <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray40", "NT" = "gray20")) +
#  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_minimal(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )

smalldotplotgray
smalldotplotbw
smalldotplotlight
smalldotplotminimal

In [ ]:
## saving the plot

bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
#   # Add vertical lines for null identity thresholds
#   geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
#   geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray40", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_minimal(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )

#bigdotplot

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplot_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 24, height = 18, units = "in", limitsize = FALSE)


## with family labels
bigdotplot2 <-  ggplot(plot_data, aes(x = pident_NTorNR, y = family_cluster, color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
#   # Add vertical lines for null identity thresholds
#   geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
#   geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  scale_color_manual(values = c("NR" = "gray40", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_minimal(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = "Family | Cluster",
       color = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplot_withfam_Mar",Sys.Date(),".pdf", sep=""), bigdotplot2, width = 24, height = 18, units = "in", limitsize = FALSE)



In [ ]:
## also with color & theme_grey    scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +


bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_gray(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )

#bigdotplot

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolor_Mar",Sys.Date(),".png", sep=""), bigdotplot, width = 24, height = 18, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolor_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 24, height = 18, units = "in", limitsize = FALSE)


### with vertical lines
bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  # Add vertical lines for null identity thresholds
  geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
  geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_gray(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )


ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolorwithlines_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 24, height = 18, units = "in", limitsize = FALSE)


## with family labels
bigdotplot2 <-  ggplot(plot_data, aes(x = pident_NTorNR, y = family_cluster, color = analysis_used, shape = analysis_used)) +
  geom_point(size = 6, alpha = 0.2) +
#   # Add vertical lines for null identity thresholds
#   geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
#   geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_gray(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = "Family | Cluster",
       color = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolor_withfam_Mar",Sys.Date(),".pdf", sep=""), bigdotplot2, width = 24, height = 18, units = "in", limitsize = FALSE)


In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown

In [ ]:
## NOW ANOTHER SET AFTER FILTERING TO ONLY KNOWN & NOVEL...
## need to go back to this thresholded_hit_virusesabundantfewcols2_tohighlight

thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown


thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown <- thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown %>%
  mutate(
    Category = factor(Category, levels = c(
      "Novel",
      "Known"
    ))
  )

# Step 1: Modify the tax_family_NTorNR column
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlight_novelknown %>% replace_na(list(tax_family_NTorNR = "(Adomaviruses)"))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Salmovirus WFRC1",
                                     "(Rhabdo-like)", tax_family_NTorNR))
## repeat for more:
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Guangdong catfish astro-like virus",
                                     "(Astro-like)", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Fish-associated picorna-like virus 3",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(taxname_lca_NTorNR == "Astyanax tetra cavefish adintovirus",
                                     "Adintoviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Salmovirus WFRC1",
                                     "(Rhabdo-like)", tax_family_NTorNR))
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Guangdong catfish astro-like virus",
                                     "(Astro-like)", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Fish-associated picorna-like virus 3",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Astyanax tetra cavefish adintovirus",
                                     "Adintoviridae", tax_family_NTorNR))

## a few more edge cases
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Coleura bat picornavirus",
                                     "Picornaviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Zebrafish picornavirus 2",
                                     "Picornaviridae", tax_family_NTorNR))

# thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
#   mutate(tax_family_NTorNR = if_else(clusterLCA == "Carp edema virus",
#                                      "Poxviridae", tax_family_NTorNR))

thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(tax_family_NTorNR = if_else(clusterLCA == "Fish-associated parvo-like hybrid virus",
                                     "Parvoviridae", tax_family_NTorNR))

#Pimephales minnow adintovirus
#Sprivivirus cyprinus Rhabdoviridae

## to Parvo-like
# Fish-associated parvo-like hybrid virus
# Fish-associated parvo-like hybrid virus

## check order
thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR[1]
unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR)



In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlightforplot <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>% 
  ungroup() %>% 
  mutate(has_parentheses = str_detect(tax_family_NTorNR, "\\(")) %>%
  arrange(Category, has_parentheses, tax_family_NTorNR, clusterLCA_curated) %>%
  select(-has_parentheses)

unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$tax_family_NTorNR)
unique(thresholded_hit_virusesabundantfewcols2_tohighlightforplot$clusterLCA_curated)


In [ ]:

## UPDATE - MOVING INTENTIONALLY ADMINISTERED KNOWN VIRUSES TO END:

thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev <- thresholded_hit_virusesabundantfewcols2_tohighlightforplot %>%
  mutate(
    sort_order = case_when(
      clusterLCA_curated == "Cyprinid herpesvirus 3" ~ 1,
      clusterLCA_curated == "Redspotted grouper nervous necrosis virus" ~ 2,
      clusterLCA_curated == "Spring viraemia of carp virus" ~ 3,
      TRUE ~ 0
    )
  ) %>%
  mutate(has_parentheses = str_detect(tax_family_NTorNR, "\\(")) %>%
  arrange(Category, sort_order, has_parentheses, tax_family_NTorNR, clusterLCA_curated) %>%
  select(-sort_order) %>%
  select(-has_parentheses) %>%
  mutate(clusterLCA_curated = factor(clusterLCA_curated, levels = rev(unique(clusterLCA_curated))))

thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev$clusterLCA_curated[1]

## rename NTclustered to just NT
thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev <- thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev %>%
  mutate(analysis_used = if_else(analysis_used == "NTclustered",
                                     "NT", analysis_used))


In [ ]:
## actual plot
# Step 4: Plot
## first small for visualizing
ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
                     aes(x = pident_NTorNR, y = clusterLCA_curated, color = analysis_used)) +
  geom_point(size = 1.2, alpha = 0.2) +
#  scale_color_manual(values = c("NR" = "red", "NTclustered" = "blue")) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_x_continuous(limits = c(0, 100)) +
  theme_grey(base_size = 12) +
  guides(color = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database") +
  theme(
    axis.text.y = element_text(size = 6),
    legend.position = "right"
  )


In [ ]:
## saving the plot instead

bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
#   # Add vertical lines for null identity thresholds
#   geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
#   geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray40", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_minimal(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )

ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplot_novelknown9x9_Mar",Sys.Date(),".png", sep=""), bigdotplot, width = 18, height = 9, units = "in", limitsize = FALSE)
ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplot_novelknown9x9_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 18, height = 9, units = "in", limitsize = FALSE)


# with lines
bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
  # Add vertical lines for null identity thresholds
  geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
  geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "gray40", "NT" = "gray20")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_minimal(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )


ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplot_novelknown9x9withlines_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 18, height = 9, units = "in", limitsize = FALSE)


## with family labels - skipping this version


In [ ]:
## also with color & theme_grey    scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +

bigdotplot <- ggplot(thresholded_hit_virusesabundantfewcols2_tohighlightforplotrev,
       aes(x = pident_NTorNR, y = clusterLCA_curated, 
           color = analysis_used, shape = analysis_used)) +
#   # Add vertical lines for null identity thresholds
#   geom_vline(xintercept = 25, linetype = "dashed", color = "gray20", size = 1) +  # NT null (light gray)
#   geom_vline(xintercept = 5, linetype = "dashed", color = "gray40", size = 1) +   # NR null (dark gray)
  geom_point(size = 6, alpha = 0.2) +
  scale_color_manual(values = c("NR" = "red", "NT" = "blue")) +
  scale_shape_manual(values = c("NR" = 17, "NT" = 16)) +  # 17 = triangle, 16 = circle
  scale_x_continuous(limits = c(0, 100)) +
  theme_gray(base_size = 28) +
  guides(color = guide_legend(override.aes = list(alpha = 1)),
         shape = guide_legend(override.aes = list(alpha = 1))) +
  labs(x = "Percent Identity to nearest NCBI hit",
       y = NULL,
       color = "Database",
       shape = "Database") +
  theme(
    axis.text.y = element_text(size = 28),
    legend.position = "right"
  )


ggsave(filename = paste("taxonomy_hits_viruses_fishspecific_percentidentitydotplotcolor_novelknown_Mar",Sys.Date(),".pdf", sep=""), bigdotplot, width = 24, height = 9, units = "in", limitsize = FALSE)


## with family labels - skipping

In [ ]:
## finally - merge thresholded_hit_virusesabundantfewcols2_tohighlight
## with sequences from taxonomy_hits_viruses_nonphage_withsequenceandclusters_2025-10-09.tsv

setwd(outpathvirus)


allchunks_diamondnr_andblastntclustered_viruses_sequence <- read_tsv("taxonomy_hits_viruses_nonphage_withsequenceandclusters_2025-10-09.tsv")             
head(allchunks_diamondnr_andblastntclustered_viruses_sequence)


In [ ]:
## change wd back
setwd(workingpath)

In [ ]:
allchunks_diamondnr_andblastntclustered_viruses_sequence <- allchunks_diamondnr_andblastntclustered_viruses_sequence %>% select(contig_withLCA_withcluster,'seq.text')


head(allchunks_diamondnr_andblastntclustered_viruses_sequence)



In [ ]:
## then join
## and then export like taxonomy_hits_viruses_clusters_nosequencenotargetsforsalmon_fishassociated_sept17_feb26update.tsv

thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- left_join(thresholded_hit_virusesabundantfewcols2_tohighlight,allchunks_diamondnr_andblastntclustered_viruses_sequence)

head(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence)

thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence

In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% relocate(contigsbyLCAcluster)
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% relocate(bioprojectsbyLCAcluster)
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% relocate(Category)
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% relocate(clusterLCA_curated)
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% relocate(contig_withLCA_withcluster)

thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence


In [ ]:
#write.table(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence, file = paste0("taxonomy_hits_viruses_clusters_fishassociated.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)

## sort by Category, clusterLCA_curated, contigsbyLCAcluster
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0 <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0 <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0 %>% ungroup()
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0 <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0 %>% arrange(Category, clusterLCA_curated)
thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence0

In [ ]:
write.table(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence, file = paste0("taxonomy_hits_viruses_clusters_fishassociated_",Sys.Date(),".tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [ ]:
thresholded_hit_virusesabundantfewcols2_tohighlight_summary <- thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence %>% select(1:8) %>%  group_by(clusterLCA_curated) %>% summarize(across(everything(), first), .groups = "drop") %>% arrange(Category, clusterLCA_curated)
head(thresholded_hit_virusesabundantfewcols2_tohighlight_summary)

In [ ]:
write.table(thresholded_hit_virusesabundantfewcols2_tohighlight_summary, file = paste0("taxonomy_hits_viruses_clusters_fishassociated_summary_",Sys.Date(),".tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [ ]:
unique(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence$clusterLCA_curated)

In [ ]:
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence$clusterLCA_curated)
n_distinct(thresholded_hit_virusesabundantfewcols2_tohighlight_withsequence$taxname_lca_NTorNR)

In [ ]:
novelchk <- thresholded_hit_virusesabundantfewcols2_tohighlight %>% filter(Category == "Novel")

In [ ]:
unique(novelchk$clusterLCA_curated)



In [ ]:


n_distinct(novelchk$taxid_lca_NTorNR)

n_distinct(novelchk$taxname_lca_NTorNR)

n_distinct(novelchk$clusterLCA_curated)

In [ ]:
unique(novelchk$taxid_lca_NTorNR)

unique(novelchk$taxname_lca_NTorNR)


In [ ]:
novel_taxonomy_breakdown <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  filter(Category == "Novel") %>%
  count(clusterLCA_curated, taxname_lca_NTorNR, taxid_lca_NTorNR, name = "n_contigs") %>%
  arrange(clusterLCA_curated, desc(n_contigs))

print(novel_taxonomy_breakdown)

In [ ]:
novel_taxonomy_breakdown

In [ ]:
write_tsv(novel_taxonomy_breakdown, "novel_viruses_taxonomy_breakdown.tsv")


In [ ]:
novelknown_taxonomy_breakdown <- thresholded_hit_virusesabundantfewcols2_tohighlight %>%
  filter(Category == "Novel" | category == "Known") %>%
  count(clusterLCA_curated, taxname_lca_NTorNR, taxid_lca_NTorNR, name = "n_contigs") %>%
  arrange(clusterLCA_curated, desc(n_contigs))

print(novelknown_taxonomy_breakdown)

In [ ]:
novel_taxonomy_breakdown